In [1]:
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help

In [2]:
import warnings
warnings.filterwarnings('ignore')
import logging

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

# SQLDB

SQLDB 是基于关系数据库构建的因子库

基于关系数据库构建的因子库和数据库原始对象的对应关系：
* 整个数据库对应于因子库
* 每张数据库表对应于因子表
* 每张数据库表的字段对应于单个因子

本质上是将一个二维的因子数据矩阵挤压成具有二重索引的一维向量. 对于因子数据的访问, 内部使用标准的 SQL 查询语句完成. 

```mermaid
graph TD
    subgraph 逻辑层
        A[因子库]
        B1[因子表]
        B2[因子表]
        A --> B1
        A --> B2
        F1[因子1]
        F2[因子2]
        B1 --> F1
        B1 --> F2
    end

    subgraph 存储层
        C[关系数据库]
        D1[数据库表]
        D2[数据库表]
        C --> D1
        C --> D2
        E2[字段2]
        E1[字段1]
        D2 --> E1
        D2 --> E2
    end

    A -.-> C
    B1 -.-> D2
    F1 -.-> E1

    style A fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style C fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
```

以下代码要求有可以访问的 postgresql 数据库，且设置好了配置文件。或者执行 [import_postgres_sqldb_demo_data.py](../tools/import_postgres_sqldb_demo_data.py) 脚本生成示例数据，这要求有写入权限的 postgresql 数据库。

SQLDB 的配置文件默认位于用户目录下的 “QuantStudioConfig” 文件夹里的 "SQLDBConfig.json" 文件。通常将数据库的连接信息配置到文件里，示例如下:
```json
{
    "Name": "SQLDB",
    "DBType": "PostgreSQL",
    "DBName": "QSData",
    "IPAddr": "localhost",
    "Port": 5432,
    "User": "postgres",
    "Pwd": "123456",
    "TablePrefix": "",
    "CharSet": "utf8",
    "Connector": "default",
    "IDField": "code",
    "DTField": "datetime"
}
```

In [5]:
# 创建因子库对象并 connect
from QuantStudio.Factor.SQLDB import SQLDB

FDB = SQLDB().connect()
print(qs_help(FDB))

类型: SQLDB
模块: QuantStudio.Factor.SQLDB
QS 对象类型: 因子库
QS 对象名称: SQLDB
QSID: 49317db5708483622a3db83806e41c836260de17b2e8e113f6b0e8e095592a52
参数集:
    * Name(名称): <class 'str'>, 默认值 'SQLDB', 当前取值: 'SQLDB'
    * DBType(数据库类型): typing.Literal['MySQL', 'SQL Server', 'Oracle', 'PostgreSQL'], 默认值 'MySQL', 当前取值: 'PostgreSQL'
    * DBName(数据库名): <class 'str'>, 默认值 'Scorpion', 当前取值: 'KDB_dev'
    * IPAddr(IP地址): <class 'str'>, 默认值 '127.0.0.1', 当前取值: 'localhost'
    * Port(端口): <class 'int'>, 默认值 3306, 当前取值: 5432
    * User(用户名): <class 'str'>, 默认值 'root', 当前取值: 'shzq'
    * TablePrefix(表名前缀): <class 'str'>, 默认值 '', 当前取值: ''
    * CharSet(字符集): typing.Literal['utf8', 'utf8mb4', 'gbk', 'gb2312', 'gb18030', 'cp936', 'big5'], 默认值 'utf8', 当前取值: 'utf8'
    * Connector(连接器): typing.Literal['default', 'cx_Oracle', 'pymssql', 'mysql.connector', 'pymysql', 'psycopg2', 'pyodbc'], 默认值 'default', 当前取值: 'default'
    * ConnRetryNum(连接重试次数): <class 'int'>, 默认值 3, 当前取值: 3
    * ConnIntervalSeconds(连接重试间隔): <class

In [6]:
# 获取因子库中的因子表列表
print(FDB.TableNames[:5])

['_stock_cn_report', 'index_cn_day_bar', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry']


# 因子表

In [7]:
# 获取因子表对象
FT = FDB.getTable("stock_cn_day_bar", args={"LookBack": 0})
print(qs_help(FT))

类型: SQL_WideTable
模块: QuantStudio.Factor.FactorUtils
QS 对象类型: 计算节点-因子表
QS 对象名称: stock_cn_day_bar
QSID: 6781381763a4f0a70531a88058b70feb6b59ec8de59e0bf502bce53ddf7106d3
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: 'stock_cn_day_bar'
    * TableType(因子表类型): typing.Literal['WideTable'], 默认值 'WideTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'WideTable'
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: 'datetime'
    * AdditionalCondition(附加条件): <class 'dict'>, 默认值 {}, 当前取值: {}
    * LookBack(回溯天数): typing.Union[int, float], 默认值 0, 缺失填充回溯的天数, 0 表示不回溯填充, 当前取值: 0
    * PublDTField(公告时点字段): typing.Optional[str], 默认值 None, 用作公告时点的字段名, 默认值 None 表示内部自动判断, 如果非 None, 表示考虑数据的公布时点, 即某个时点所能获取的数据必须保证其在公告时点和截止时点之后, 当前取值: None
    * MultiMapping(多重映射): <class 'bool'>, 默认值 False, 是否为高维数据, 即时点和 ID 两个维度无法唯一索引单个数据, 默认形成的数据在单个时点单个 ID 处以 list 形式表达, 当前取值: False
    * Operator(算子): typing.Optional[typing.Callable], 默认值 None, 对于单个时点单个 ID 处的数据 appl

In [8]:
# 因子列表
FT = FDB.getTable("stock_cn_day_bar")
print(FT.FactorNames)

['amount', 'close', 'code', 'datetime', 'high', 'low', 'open', 'volume']


## 读取数据

In [10]:
# 因子表读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

FT = FDB.getTable("stock_cn_day_bar", args={"LookBack": 0})
Data = FT.readData(factor_names=["close", "open"], ids=IDs, dts=DTs)
print("因子表数据")
print(Data)

因子表数据
<class 'QuantStudio.Core.QSObject.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 2 (minor_axis)
Items axis: close to open
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-05 00:00:00
Minor_axis axis: 000001.SZ to 000002.SZ


# 因子

In [11]:
# 获取因子对象
F = FT.getFactor("close")
print(qs_help(F))

类型: Factor
模块: QuantStudio.Factor.Factor
QS 对象类型: 计算节点-因子
QS 对象名称: close
QSID: 28982d16234c25632a97f6c34df2eae4a35db1236185a23969a935950b725eeb
参数集:
    * Name(名称): <class 'str'>, 默认值 'Factor', 当前取值: 'close'
    * Meta(元信息): <class 'dict'>, 默认值 {}, 当前取值: {}
    * SectionIDs(截面ID): typing.Optional[typing.List[str]], 默认值 None, 当前取值: None
    * CalcDTRuler(计算时点标尺): typing.Optional[typing.List[datetime.datetime]], 默认值 None, 当前取值: None
说明文档:
    因子对象
    因子可看做 DataFrame(index=[时点], columns=[ID])
    时点数据类型是 datetime, ID 的数据类型是 str


## 读取数据

In [12]:
# 因子读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = F.readData(ids=IDs, dts=DTs)
print(Data)

                     000001.SZ  000002.SZ
2025-01-01 00:00:00   8.115185   4.760840
2025-01-02 00:00:00   0.352198   1.806606
2025-01-03 00:00:00   6.019437   0.633690
2025-01-04 00:00:00   3.936297   3.755494
2025-01-05 00:00:00   4.884425   1.342670
